Akua Duffour,
Cory Parker,
Paola Marsal

***Load, Clean, and Filter Dataset Cities to European Cities Only***

In [0]:
#loading the raw tables from the schema into PySpark DataFrames
cities_df = spark.read.table("main.alcala_voyages.worldwide_travel_cities")
hotels_df = spark.read.table("main.alcala_voyages.europe_hotel_reviews")

#filter the cities dataset to european cities only
# The dataset has a 'country' column. We will filter for the 6 countries 
# matching your European hotel dataset to keep them perfectly aligned.
europe_countries = ["United Kingdom", "France", "Spain", "Austria", "Italy", "Netherlands"]
filtered_cities_df = cities_df.filter(cities_df["country"].isin(europe_countries))

#clean missing data values to drop empty values
clean_cities_df = filtered_cities_df.dropna(subset=["city", "country"])
clean_hotels_df = hotels_df.dropna(subset=["Hotel_Name", "Positive_Review", "Negative_Review"])

#save the filtered and cleaned cities to new table clean_europe_cities
clean_cities_df.write.format("delta").mode("overwrite").saveAsTable("main.alcala_voyages.clean_europe_cities")

#preview to verify it worked
display(spark.read.table("main.alcala_voyages.clean_europe_cities").limit(5))

***Build Vector Index Search on Hotel Reviews for RAG***

In [0]:
#aggregate hotel reviews for AI RAG tool to search efficiently
from pyspark.sql import functions as F

#group by the unique hotel name and combine all of its text reviews together
#F.collect_list to gather all reviews, F.concat_ws to join them with a space
aggregated_hotels_df = clean_hotels_df.groupBy("Hotel_Name", "Hotel_Address").agg(
    F.concat_ws(" ", F.collect_list("Positive_Review")).alias("All_Positive_Reviews"),
    F.concat_ws(" ", F.collect_list("Negative_Review")).alias("All_Negative_Reviews"),
    F.avg("Average_Score").alias("Final_Average_Score")
)

#combine the positive and negative review blocks into one master text column for the vector indexing
aggregated_hotels_df = aggregated_hotels_df.withColumn(
    "Hotel_Full_Description",
    F.concat_ws(" ", 
        F.lit("Hotel Name:"), F.col("Hotel_Name"),
        F.lit("Address:"), F.col("Hotel_Address"),
        F.lit("Positive Feedback:"), F.col("All_Positive_Reviews"),
        F.lit("Negative Feedback:"), F.col("All_Negative_Reviews")
    )
)

#save aggregated data to new hotel data table
aggregated_hotels_df.write.format("delta").mode("overwrite").saveAsTable("main.alcala_voyages.clean_aggregated_hotels")

#preview to verify it worked
display(spark.read.table("main.alcala_voyages.clean_aggregated_hotels").select("Hotel_Name", "Final_Average_Score").limit(15))

***Create the Databricks Vector Search Index***

In [0]:
#enable change data feed (CDF) on the clean_aggregated_hotels table
#this is a mandatory requirement for databricks vector search to track our data
spark.sql("""
    ALTER TABLE main.alcala_voyages.clean_aggregated_hotels 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Change Data Feed successfully enabled! Your table is now ready for Vector search!!")